# Compare Rust horizontal mapping with RAS Mapper

Generate native **horizontal** depth/WSE stored maps and compare them with the Rust horizontal engine, using the same strict `depth > 0.5 ft` filter. The library compares matching terrain pixels without resampling the reference and verifies/restores the original `.rasmap` bytes after native mapping.

Use computed, staged projects from two large eBFE models and one RasExamples model. Native stored-map generation can take minutes and write large rasters; it does not rerun the hydraulic solver. A bounded AOI controls Rust work and comparison size; native stored-map generation still maps the selected plan.

This source notebook is an execution template. Run-specific evidence is written under `WORK`. Agreement applies to the selected 2D cells, terrain, rendering mode, timesteps and cutoff, not native sloping, 1D or structure rendering.

## Install and select the models

In a terminal, install `ras-commander[fim]`, the matching **0.2.0** engine wheel, `matplotlib` and a notebook kernel. A supported HEC-RAS mapping runtime must also be installed. The engine itself uses terrio without GDAL; Python still prepares ownership and writes GIS products.

| Corpus model | Selected plan / mesh | Physical cells observed in the retained case | Why include it |
|---|---|---:|---|
| eBFE Spring River, HUC11010010 | 02 / Spring | 422,715 | Large mesh; 761-million-pixel source terrain; EPSG:3433 |
| eBFE Spring Creek, HUC12040102 | 01 / SPR_Mesh | 340,163 | Independent large mesh; 447-million-pixel source terrain; EPSG:2278 |
| RasExamples BaldEagleCrkMulti2D | 02 / BaldEagleCr | 87,039 | Dam-break wetting front; multi-source terrain; EPSG:2271 |

The eBFE retained results were produced by HEC-RAS6.1 and5.0.7 respectively; distinguish the solver version from the mapper version used here. BaldEagle's public source may need a separate compute through `RasCmdr.compute_plan` before this workflow. Existing complete results can be supplied instead. All three selected cases use feet for elevation; do not infer vertical units from the horizontal CRS alone.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import sys

USE_LOCAL_SOURCE = False
if USE_LOCAL_SOURCE:
    repo_root = Path.cwd().resolve()
    if repo_root.name == "examples":
        repo_root = repo_root.parent
    sys.path.insert(0, str(repo_root))

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from ras_commander import RasExamples, RasFim, RasTerrain, init_ras_project

WORK = Path(os.environ.get("RAS_COMMANDER_FIM_COMPARISON_ROOT", "working/fim-horizontal-comparison")).resolve()
WORK.mkdir(parents=True, exist_ok=True)
CONFIG = Path(os.environ.get("RAS_COMMANDER_FIM_CORPUS_CONFIG", WORK / "cases.json"))
MAPPER_VERSION = "7.0.1"
MIN_DEPTH = 0.5                 # feet for these selected models
THREADS = 4
MAX_WORKING_MIB = 2048


## Acquire inputs and configure a run

The acquisition function below uses the public source adapters. Call it only when you need to download/organize the corpus; it is not needed for retained computed copies. Configure `RAS_COMMANDER_EBFE_ROOT` as appropriate for archive staging. Acquisition does not establish that a plan has computed results.

Create `cases.json` with two or three entries. Each identifies a computed project, plan, mesh and the exact registered terrain. For example:

```json
[
  {"model":"Spring River", "project_file":"prepared/SpringRiver/RAS Model/Spring_BLE.prj", "plan":"02", "mesh":"Spring", "terrain":"Terrain (2)"},
  {"model":"Spring Creek", "project_file":"prepared/SpringCreek/RAS Model/Spring.prj", "plan":"01", "mesh":"SPR_Mesh", "terrain":"Terrain(BurnDEMv2)"},
  {"model":"BaldEagle", "project_file":"prepared/BaldEagle/BaldEagleDamBrk.prj", "plan":"02", "mesh":"BaldEagleCr", "terrain":"Terrain50"}
]
```

Paths resolve against the config folder. The workflow copies each whole project into a new local staging folder, preserving relative terrain/projection references. To use an existing isolated copy inside `WORK`, add `"staged": true`. Never mark a shared corpus source as staged. Optional `bbox` contains model-coordinate bounds; otherwise a reproducible 10,000-foot square around the middle wet cell is used.

In [ ]:
def acquire_sources(download_root, output_root):
    output_root = Path(output_root)
    river = RasExamples.organize_ebfe_model("spring-river", output_path=output_root,
                                          download_root=Path(download_root))
    creek = RasExamples.organize_ebfe_model("spring-creek", output_path=output_root,
                                          download_root=Path(download_root))
    eagle = RasExamples.extract_project("BaldEagleCrkMulti2D", output_path=output_root)
    return {"spring-river": river, "spring-creek": creek, "bald-eagle": eagle}

cases = json.loads(CONFIG.read_text(encoding="utf-8"))
assert 2 <= len(cases) <= 3, "Select two or three corpus cases"
for case in cases:
    source = Path(case["project_file"])
    if not source.is_absolute():
        source = CONFIG.parent / source
    source = source.resolve()
    if not source.is_file():
        raise FileNotFoundError(source)
    if case.get("staged", False):
        if not source.is_relative_to(WORK):
            raise ValueError("A staged project must be inside the isolated WORK folder")
        case["staged_project"] = source
    else:
        destination = WORK / "models" / case["model"].replace(" ", "_")
        # Preflight expanded paths before copying; preserve the source tree.
        longest = max(len(str(destination / p.relative_to(source.parent)))
                      for p in source.parent.rglob("*") if p.is_file())
        if longest > 240:
            raise ValueError("Choose a shorter configured WORK path before staging")
        shutil.copytree(source.parent, destination)  # refuses an existing destination
        case["staged_project"] = destination / source.name
pd.DataFrame([{k:v for k,v in c.items() if k != "staged_project"} for c in cases])


## Export the exact native terrain and compare

Use the selected registered terrain, including native modifications. A failed export or incompatible lattice raises; the workflow does not replace it with an arbitrary source TIFF. `compare_with_rasmapper` owns native execution, Rust mapping, thresholded metrics, the project lock and `.rasmap` restoration.

Maximum WSE is selected here. To compare saved times, provide cell WSE columns from the same plan/mesh and pass `profiles={column: exact_native_timestamp}` to the same API. Native timestamps can be listed with `RasProcess.get_plan_timestamps`. The comparison records the caller's supplied pairing; it does not guess nearby times.

In [ ]:
results = []
for i, case in enumerate(cases):
    project = init_ras_project(case["staged_project"], MAPPER_VERSION, ras_object="new",
                               hide_intro=True, load_results_summary=False)
    mapping = project.rasmap_df.iloc[0]
    if mapping["rasmap_status"] not in ("parsed", "parsed_with_errors"):
        raise ValueError("The staged rasmap must parse successfully")
    inputs = [mapping["projection_path"], *mapping["terrain_hdf_path"]]
    for path in inputs:
        resolved = Path(path).resolve()
        if not resolved.is_file() or not resolved.is_relative_to(project.project_folder.resolve()):
            raise ValueError("Stage all referenced projection and terrain files inside the project")
    row = project.plan_df.loc[project.plan_df["plan_number"] == case["plan"]].iloc[0]
    hdf_path = Path(str(row["full_path"]) + ".hdf")
    cells = RasFim.get_cells_from_plan_hdf(hdf_path, mesh_name=case["mesh"])
    wet = cells.loc[cells["wsel_max"].notna()]
    if wet.empty:
        raise ValueError(f"No wet cell maxima in {case['model']}")
    point = wet.geometry.iloc[len(wet)//2].representative_point()
    bbox = case.get("bbox", [point.x-5000, point.y-5000, point.x+5000, point.y+5000])
    out = WORK / "runs" / f"{i:02d}"
    out.mkdir(parents=True, exist_ok=False)
    terrain = out / "native-terrain.tif"
    exported = RasTerrain.export_rasmapper_terrain(
        project.project_folder, terrain, terrain_name=case["terrain"], extent=bbox,
        rasterize_modifications=True, hecras_version=MAPPER_VERSION,
        ras_object=project, timeout_seconds=1800,
    )
    if not exported:
        raise RuntimeError(f"Native terrain export failed: {exported}")
    comparison = RasFim.compare_with_rasmapper(
        case["plan"], terrain, out / "comparison", cells=cells,
        profiles={"wsel_max": "Max"}, mesh_name=case["mesh"], terrain_name=case["terrain"],
        bbox=bbox, min_depth=MIN_DEPTH, threads=THREADS,
        max_working_mib=MAX_WORKING_MIB, ras_object=project,
        ras_version=MAPPER_VERSION, timeout=1800,
    )
    comparison.insert(0, "model", case["model"])
    comparison["physical_cells"] = len(cells)
    comparison["rust_batch_seconds"] = comparison.attrs["candidate_batch_seconds"]
    assert comparison.attrs["rasmap_restored"]
    results.append(comparison)
    display(comparison[["model", "physical_cells", "csi", "extra_wet", "missing_wet",
                        "depth_max_abs", "wse_exact_fraction", "terrain_max_abs"]])
    del cells, wet

summary = pd.concat(results, ignore_index=True)
summary.to_csv(WORK / "corpus-comparison.csv", index=False)
summary


## Inspect one comparison spatially

The report uses the strict cutoff independently on both depth rasters. It separates missing native WSE from finite WSE pairs and reports the native implied-terrain residual (`native WSE - native depth - terrain`). A nonzero residual is diagnostic; it does not identify its cause automatically.

The plot below uses integer-offset reads only, after the API's lattice checks. Inspect wet/dry disagreements alongside depth and WSE metrics. Near-zero differences on these selected AOIs do not establish whole-model, sloping, structure or mixed1D/2D parity.

In [ ]:
from rasterio.windows import Window

row = summary.iloc[-1]
with rasterio.open(row["depth_tif"]) as candidate, rasterio.open(row["reference_depth_tif"]) as reference:
    a = candidate.read(1, masked=True).filled(np.nan)
    col = round((candidate.transform.c-reference.transform.c)/reference.transform.a)
    line = round((candidate.transform.f-reference.transform.f)/reference.transform.e)
    b = reference.read(1, window=Window(col,line,candidate.width,candidate.height),
                       boundless=True, masked=True).astype("float32").filled(np.nan)
    a = np.where(a > np.float32(MIN_DEPTH), a, np.nan)
    b = np.where(b > np.float32(MIN_DEPTH), b, np.nan)
fig, axes = plt.subplots(1,3,figsize=(14,5),layout="constrained")
for axis,data,title in zip(axes,[b,a,a-b],["Native horizontal", "Rust horizontal", "Depth difference"]):
    image = axis.imshow(data, cmap="RdBu_r" if title=="Depth difference" else "Blues")
    axis.set_title(title);axis.set_axis_off()
    fig.colorbar(image,ax=axis,label="feet")
plt.show()


## Interpret timings and preserve evidence

`native_seconds` measures each native stored-map call. `rust_batch_seconds` measures the entire candidate batch, including preparation and GIS writing. These include different stages and are not a controlled speed ratio. For throughput measurements, stage identical inputs before timing, use equal CPU budgets, interleave repeated runs, retain warm-up/cache policy, and require numerical equality between Rust and the explicit NumPy reference.

Retain `comparison.json`, `comparison.csv`, the terrain export receipt and original raster products. The report binds the native runtime, terrain identity, threshold and restored `.rasmap` hashes. Do not commit model data or large generated rasters. Native horizontal construction follows the [HEC-RAS6.6 Mapping Options](https://www.hec.usace.army.mil/confluence/rasdocs/rmum/6.6/mapping-results/mapping-options); RAS Commander is an independent implementation, and these checks do not imply HEC endorsement.